# Tokenizer
Patch embedding and fixed sinusoidal positional encodings.

In [ ]:
import math
import torch
import torch.nn as nn
import sys, os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config

In [ ]:
class PatchEmbedding(nn.Module):
    """Splits a padded flux array into fixed-size patches and projects each to d_model.

    Args:
        patch_size (int): Number of flux pixels per patch.
        d_model (int): Output embedding dimension.
    """

    def __init__(self, patch_size=config.PATCH_SIZE, d_model=config.D_MODEL):
        super().__init__()
        self.patch_size = patch_size
        self.projection = nn.Linear(patch_size, d_model)

    def forward(self, flux):
        """Project a batch of padded spectra into patch token embeddings.

        Args:
            flux (torch.Tensor): Shape (B, PADDED_LEN).

        Returns:
            torch.Tensor: Shape (B, N_PATCHES, d_model).
        """
        B = flux.shape[0]
        patches = flux.reshape(B, -1, self.patch_size)
        return self.projection(patches)

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    """Compute fixed sinusoidal positional encodings.

    Args:
        seq_len (int): Number of positions.
        d_model (int): Embedding dimension.

    Returns:
        torch.Tensor: Shape (seq_len, d_model).
    """
    position = torch.arange(seq_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe